# Notebook 06 â€” Entity Matching ML

Tasks:
- feature engineering
- entity-level split
- Logistic Regression
- Random Forest
- optional gradient boosting
- threshold tuning
- error analysis

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

OUTPUT_DIR = "D:/data viewer/data/processed"

## Load Features

In [ ]:
features = pd.read_csv(f"{OUTPUT_DIR}/pairwise_features.csv")
features = features.fillna(0)

print(f"Total pairs: {len(features)}")
print(f"Class balance:")
print(features['label'].value_counts())
features.head()

## Load Real Dataset (UCI Donation)

Dataset: Record Linkage Comparison Patterns (UCI ML Repository)
Source: https://archive.ics.uci.edu/ml/machine-learning-databases/00210/donation.zip
License: Public domain (research dataset)
Records: 5,749,132 record pairs with match/non-match labels
Usage: Entity resolution training with real data

In [ ]:
# Load real dataset from UCI donation
import os
donation_path = 'D:/data viewer/data/raw/donation/block_1/block_1.csv'
if os.path.exists(donation_path):
    donation = pd.read_csv(donation_path, nrows=100000)
    donation['is_match'] = donation['is_match'].astype(int)
    print(f'Donation dataset: {donation.shape}')
    print(f'Class balance:')
    print(donation['is_match'].value_counts())
    print(f'Columns: {list(donation.columns)}')
    donation.head()
else:
    print('Donation dataset not found. Run data download first.')

## Entity-Level Split

In [ ]:
# Entity-level split: ensure same entity doesn't appear in both train and test
pairs = pd.read_csv(f"{OUTPUT_DIR}/entity_pairs.csv")
features = features.merge(pairs[['pair_id', 'master_id_a', 'master_id_b']], on='pair_id')

unique_entities = set(features['master_id_a'].unique()) | set(features['master_id_b'].unique())
entity_list = list(unique_entities)
train_entities, test_entities = train_test_split(entity_list, test_size=0.3, random_state=42)

train_mask = features['master_id_a'].isin(train_entities) & features['master_id_b'].isin(train_entities)
test_mask = features['master_id_a'].isin(test_entities) & features['master_id_b'].isin(test_entities)

train_df = features[train_mask].copy()
test_df = features[test_mask].copy()

print(f"Train pairs: {len(train_df)} (entities: {len(train_entities)})")
print(f"Test pairs: {len(test_df)} (entities: {len(test_entities)})")
print(f"\nTrain class balance:")
print(train_df['label'].value_counts())
print(f"\nTest class balance:")
print(test_df['label'].value_counts())

## Train Models

In [ ]:
feature_cols = ['customer_similarity', 'installer_similarity', 'address_similarity', 'city_match', 'state_match', 'capacity_diff', 'status_match']

X_train = train_df[feature_cols]
y_train = train_df['label']
X_test = test_df[feature_cols]
y_test = test_df['label']

# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print("Logistic Regression:")
print(f"  Precision: {precision_score(y_test, y_pred_lr):.3f}")
print(f"  Recall: {recall_score(y_test, y_pred_lr):.3f}")
print(f"  F1: {f1_score(y_test, y_pred_lr):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.3f}")

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("\nRandom Forest:")
print(f"  Precision: {precision_score(y_test, y_pred_rf):.3f}")
print(f"  Recall: {recall_score(y_test, y_pred_rf):.3f}")
print(f"  F1: {f1_score(y_test, y_pred_rf):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.3f}")

## Confusion Matrices

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(cm_lr, interpolation='nearest', cmap='Blues')
axes[0].set_title('Logistic Regression')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm_lr[i, j], ha='center', va='center')

im2 = axes[1].imshow(cm_rf, interpolation='nearest', cmap='Blues')
axes[1].set_title('Random Forest')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, cm_rf[i, j], ha='center', va='center')

plt.tight_layout()
plt.show()

## Threshold Tuning

In [ ]:
thresholds = np.arange(0.3, 1.0, 0.05)
results = []

for thresh in thresholds:
    y_pred = (y_prob_rf >= thresh).astype(int)
    results.append({
        'threshold': thresh,
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(results_df['threshold'], results_df['precision'], marker='o', label='Precision')
ax.plot(results_df['threshold'], results_df['recall'], marker='s', label='Recall')
ax.plot(results_df['threshold'], results_df['f1'], marker='^', label='F1')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Tuning (Random Forest)')
ax.legend()
ax.grid(True)
plt.show()

## Error Analysis

In [ ]:
# False positives and false negatives
test_df['pred'] = y_pred_rf
test_df['prob'] = y_prob_rf

fp = test_df[(test_df['label'] == 0) & (test_df['pred'] == 1)]
fn = test_df[(test_df['label'] == 1) & (test_df['pred'] == 0)]

print(f"False positives: {len(fp)}")
print(f"False negatives: {len(fn)}")
print(f"\nFalse positive rate: {len(fp)/(len(fp)+len(test_df[(test_df['label']==0)&(test_df['pred']==0)])):.3f}")
print(f"False negative rate: {len(fn)/(len(fn)+len(test_df[(test_df['label']==1)&(test_df['pred']==1)])):.3f}")

print(f"\nSample false positives:")
print(fp[['pair_id', 'customer_similarity', 'installer_similarity', 'address_similarity', 'prob']].head(10))

print(f"\nSample false negatives:")
print(fn[['pair_id', 'customer_similarity', 'installer_similarity', 'address_similarity', 'prob']].head(10))

## Supervised ML on Real Dirty Data + TF-IDF

Train Logistic Regression and Random Forest on dirty Walmart-Amazon pairs with Jaro-Winkler + TF-IDF cosine + brand/category/price features. Compare against the clean version of the same test pairs.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RAW_DIR = "D:/data viewer/data/raw"
DM_DIR = f"{RAW_DIR}/deepmatcher"

def dm_norm(s):
    if pd.isna(s):
        return ""
    return str(s).strip().lower()

dirty_train = pd.read_csv(f"{DM_DIR}/dirty_walmart_amazon/train.csv")
dirty_test = pd.read_csv(f"{DM_DIR}/dirty_walmart_amazon/test.csv")
clean_test = pd.read_csv(f"{DM_DIR}/walmart_amazon/test.csv")
print(f"dirty train: {dirty_train.shape}, dirty test: {dirty_test.shape}, clean test: {clean_test.shape}")

titles = pd.concat([dirty_train["left_title"], dirty_train["right_title"]]).fillna("").astype(str).str.lower()
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2)).fit(titles)

def dm_features(df):
    lt = df["left_title"].fillna("").astype(str).str.lower()
    rt = df["right_title"].fillna("").astype(str).str.lower()
    cos = [float(cosine_similarity(tfidf.transform([a]), tfidf.transform([b]))[0, 0]) for a, b in zip(lt, rt)]
    pa = pd.to_numeric(df["left_price"], errors="coerce")
    pb = pd.to_numeric(df["right_price"], errors="coerce")
    out = pd.DataFrame({
        "title_tfidf_cosine": cos,
        "title_exact": (lt == rt).astype(int),
        "brand_match": (df["left_brand"].fillna("").astype(str).str.lower() == df["right_brand"].fillna("").astype(str).str.lower()).astype(int),
        "category_match": (df["left_category"].fillna("").astype(str).str.lower() == df["right_category"].fillna("").astype(str).str.lower()).astype(int),
        "price_diff": (pa - pb).abs().fillna(0),
    })
    return out

X_tr, y_tr = dm_features(dirty_train), dirty_train["label"].values
X_dte, y_dte = dm_features(dirty_test), dirty_test["label"].values
X_cte, y_cte = dm_features(clean_test), clean_test["label"].values
print("Feature sample:")
print(X_tr.head(3))

In [ ]:
dm_models = {
    "logistic_regression": LogisticRegression(random_state=42, max_iter=1000),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
}
dm_report = []
for mname, model in dm_models.items():
    model.fit(X_tr, y_tr)
    for dname, Xe, ye in [("dirty_test", X_dte, y_dte), ("clean_test", X_cte, y_cte)]:
        yp = model.predict(Xe)
        dm_report.append({
            "model": mname, "eval_set": dname,
            "precision": round(precision_score(ye, yp, zero_division=0), 3),
            "recall": round(recall_score(ye, yp, zero_division=0), 3),
            "f1": round(f1_score(ye, yp, zero_division=0), 3),
            "roc_auc": round(roc_auc_score(ye, model.predict_proba(Xe)[:, 1]), 3),
        })

dm_report_df = pd.DataFrame(dm_report)
print(dm_report_df.to_string(index=False))
dm_report_df.to_csv(f"{OUTPUT_DIR}/dirty_er_ml.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR}/dirty_er_ml.csv")